# Part 2 - MapReduce and Visualisation

## Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path
import multiprocessing
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time
import io, contextlib
import json
import pandas as pd
import pyspark

os.environ["JAVA_HOME"] = subprocess.run(
    ["/usr/libexec/java_home", "-v", "17"], capture_output=True, text=True).stdout.strip()
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("python ", sys.version.split()[0])
print("pyspark", pyspark.__version__)
print("pandas ", pd.__version__)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

In [ ]:
# Paths

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
LOCAL = REPO / "data"
LOCAL_RAW = LOCAL / "working"         
LOCAL_CUR = LOCAL / "filtered"
LOCAL_OUT = REPO / "output"
EVENTS = LOCAL / "spark-events"
DRIVE_OUT = REPO / "output"           
DRIVE_FIG = REPO / "figures"
for p in (LOCAL_CUR, LOCAL_OUT, EVENTS, DRIVE_OUT, DRIVE_FIG):
    p.mkdir(parents=True, exist_ok=True)

CORES = multiprocessing.cpu_count()
DRIVER_MEM = "12g"                  
WINDOW = [(y, m) for y in range(2022, 2026) for m in range(1, 13)]

MIN_NIGHT_TRIPS, MIN_PEAK_TRIPS = 100, 500
AIRPORTS = [132, 138]

print("repo", REPO, ",cores", CORES)

In [ ]:
# Ran notebook locally

names   = [f"yellow_tripdata_{y}-{m:02d}.parquet" for y, m in WINDOW]
staged  = [str(LOCAL_RAW / n) for n in names if (LOCAL_RAW / n).exists()]
missing = [n for n in names if not (LOCAL_RAW / n).exists()]
nbytes  = sum(Path(p).stat().st_size for p in staged)

print(f"{len(staged)}/48 months present")
print(f"working set: {nbytes / 2**30:.2f} GiB")
print(f"missing: {missing or 'none'}")

In [ ]:
# Spark session builder

spark = (SparkSession.builder
    .appName("MIT805-Part2-Congestion")
    .master(f"local[{CORES}]")
    .config("spark.driver.memory", DRIVER_MEM)
    .config("spark.sql.shuffle.partitions", CORES * 4)    
    .config("spark.sql.session.timeZone", "UTC")           
    .config("spark.sql.adaptive.enabled", "false")       
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", f"file://{EVENTS}")
    .config("spark.local.dir", str(LOCAL / "spark-tmp"))
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(spark.version, "cores:", CORES, "shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI:", spark.sparkContext.uiWebUrl)

## 00: Data filtering and partitioning

In [ ]:
def normalise(df):
    return df.toDF(*[c.lower() for c in df.columns])

raw = normalise(spark.read.parquet(*staged))

base = (raw
    .select(
        F.col("tpep_pickup_datetime").alias("pickup_ts"),
        F.col("tpep_dropoff_datetime").alias("dropoff_ts"),
        F.col("pulocationid").cast("int").alias("pu"),
        F.col("dolocationid").cast("int").alias("do"),
        F.col("trip_distance").cast("double").alias("miles"),
        F.col("fare_amount").cast("double").alias("fare"),
        F.col("tip_amount").cast("double").alias("tip"))
    .withColumn("duration_hr",
                F.expr("timestampdiff(SECOND, pickup_ts, dropoff_ts)") / 3600.0)
    .withColumn("speed_mph", F.expr("try_divide(miles, duration_hr)"))
    .withColumn("revenue", F.col("fare") + F.coalesce(F.col("tip"), F.lit(0.0)))
    .withColumn("hour", F.hour("pickup_ts"))
    .withColumn("daytype", F.when(F.dayofweek("pickup_ts").isin(1, 7), "weekend")
                            .otherwise("weekday")))

base.printSchema()

In [ ]:
# Data checks and filtering

CHECKS = {
    "in_window": "year(pickup_ts) between 2022 and 2025", # avoids NTZ vs zoned-literal mismatch
    "ordered": "dropoff_ts > pickup_ts",
    "duration_pos": "duration_hr > 0", # check for zero-duration trips
    "duration_ok": "duration_hr between 0.0166 and 3.0", # 1 minute to 3 hours
    "distance_ok": "miles between 0.1 and 100", # 0.1 to 100 miles
    "speed_ok": "speed_mph between 1 and 70", # speed between 1 and 70 mph
    "money_ok": "fare between 0.01 and 1000", # trip fare between 0.01 and 1000 USD
    "zones_ok": "pu between 1 and 263 AND do between 1 and 263", # 264/265 = Unknown / N.V.
}

audit = (base.select(
            F.count(F.lit(1)).alias("rows_read"),
            *[F.sum(F.when(F.expr(c), 1).otherwise(0)).alias(name) for name, c in CHECKS.items()])
         .collect()[0].asDict())
for k, v in audit.items():
    print(f"{k:14s} {v:>14,}  ({v / audit['rows_read']:.2%})")

keep  = " AND ".join(f"({c})" for c in CHECKS.values())
clean = base.filter(keep)

(clean
    .withColumn("year",  F.year("pickup_ts"))
    .withColumn("month", F.month("pickup_ts"))
    .repartition("year", "month")
    .write.mode("overwrite").partitionBy("year", "month")
    .parquet(str(LOCAL_CUR)))

filtered = spark.read.parquet(str(LOCAL_CUR))
print("filtered rows:", f"{filtered.count():,}")

In [ ]:
(filtered.groupBy("hour")
        .agg(F.count(F.lit(1)).alias("trips"),
             (F.sum("miles") / F.sum("duration_hr")).alias("speed_mph"),
             (F.sum("revenue") / F.sum("duration_hr")).alias("yield_per_hr"))
        .orderBy("hour").show(24))

print("filtered size:",
      sum(f.stat().st_size for f in LOCAL_CUR.rglob("*.parquet")) / 2**30, "GiB")

In [ ]:
SLICE = LOCAL_CUR / "year=2024" / "month=6"

SHARE = Path.home() / "Library/CloudStorage/GoogleDrive-u19031786@tuks.co.za/My Drive/NYC_Taxi_Project_SHARED"
if SHARE.exists():
    subprocess.run(f"cp -r '{SLICE}' '{SHARE}/filtered_slice_2024_06'", shell=True)
    print("copied to", SHARE)
else:
    ZIP = Path.home() / "Downloads" / "filtered_slice_2024_06.zip"
    subprocess.run(f"cd '{LOCAL_CUR}' && zip -qr '{ZIP}' 'year=2024/month=6'", shell=True)
    print("zipped to", ZIP)

schema_lines = [f"{f.name:14s} {f.dataType.simpleString()}" for f in filtered.schema.fields]
(DRIVE_OUT / "filtered_schema.txt").write_text(
    "MIT805 Part 2 filtered schema - frozen 2026-09-25\n" + "\n".join(schema_lines))
print("\n".join(schema_lines))

## 01: Baseline

In [ ]:
MIN_NIGHT_TRIPS = 100  
night = filtered.filter("hour between 1 and 4 AND daytype = 'weekday'")

ff_route = (night
    .groupBy("pu", "do")
    .agg(F.sum("miles").alias("m"),
         F.sum("duration_hr").alias("h"),
         F.count(F.lit(1)).alias("n_night"))      
    .filter(F.col("n_night") >= MIN_NIGHT_TRIPS)
    .withColumn("ff_speed", F.col("m") / F.col("h"))
    .select("pu", "do", "ff_speed", "n_night")
    .cache())

ff_zone = (night
    .groupBy("pu")
    .agg(F.sum("miles").alias("m"),
         F.sum("duration_hr").alias("h"),
         F.sum("revenue").alias("r"),
         F.count(F.lit(1)).alias("n_night"))
    .withColumn("ff_speed_zone", F.col("m") / F.col("h"))
    .withColumn("ff_yield_zone", F.col("r") / F.col("h")) # baseline $/vehicle-hour
    .select("pu", "ff_speed_zone", "ff_yield_zone", "n_night")
    .cache())
 
print("routes with a baseline:", f"{ff_route.count():,}", ",zones:", ff_zone.count())
ff_route.orderBy(F.desc("n_night")).show(5)

## 02: RQ2 - Routes

In [ ]:
MIN_PEAK_TRIPS = 500

WINDOW_COL = (F.when(F.col("daytype") == "weekend", F.lit("weekend"))
               .when(F.col("hour").between(7, 9),   F.lit("am_peak"))
               .when(F.col("hour").between(16, 18), F.lit("pm_peak"))
               .when(F.col("hour").between(10, 15), F.lit("midday"))
               .otherwise(F.lit("night")))

route_window = (filtered
    .withColumn("window", WINDOW_COL)
    .groupBy("pu", "do", "window")
    .agg(F.sum("miles").alias("m"),
         F.sum("duration_hr").alias("h"),
         F.sum("revenue").alias("rev"),
         F.count(F.lit(1)).alias("trips"))
    .filter(F.col("trips") >= MIN_PEAK_TRIPS)
    .withColumn("speed", F.col("m") / F.col("h")))

rq2 = (route_window
    .join(F.broadcast(ff_route), ["pu", "do"])
    .withColumn("tti", F.col("ff_speed") / F.col("speed"))
    .withColumn("delay_hr_total", F.col("h") - F.col("m") / F.col("ff_speed"))
    .withColumn("delay_min_per_trip", F.col("delay_hr_total") * 60 / F.col("trips"))
    .cache())

print(f"{rq2.count():,} route-window cells")

In [ ]:
subprocess.run(f"wget -q -O {LOCAL}/taxi_zone_lookup.csv "
               "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", shell=True, check=True)

zones = (normalise(spark.read.option("header", True).csv(str(LOCAL / "taxi_zone_lookup.csv")))
         .select(F.col("locationid").cast("int").alias("loc"), "zone", "borough"))

named = (rq2
    .join(F.broadcast(zones.withColumnRenamed("loc", "pu")
                           .withColumnRenamed("zone", "pu_zone")
                           .withColumnRenamed("borough", "pu_boro")), "pu")
    .join(F.broadcast(zones.withColumnRenamed("loc", "do")
                           .withColumnRenamed("zone", "do_zone")
                           .withColumnRenamed("borough", "do_boro")), "do"))

pm = named.filter("window = 'pm_peak'")

print("Worst by severity (TTI):")
pm.orderBy(F.desc("tti")).select("pu_zone", "do_zone", "trips", "speed", "ff_speed",
                                 "tti", "delay_min_per_trip").show(20, truncate=False)

print("Worst by total burden (delay hours):")
pm.orderBy(F.desc("delay_hr_total")).select("pu_zone", "do_zone", "trips",
                                            "delay_hr_total", "tti").show(20, truncate=False)

## 03: RQ1 - Revenue

In [ ]:
trip_delay = (filtered
    .join(F.broadcast(ff_route), ["pu", "do"], "left")
    .join(F.broadcast(ff_zone.select("pu", "ff_speed_zone")), ["pu"], "left")
    .withColumn("ff_eff", F.coalesce(F.col("ff_speed"), F.col("ff_speed_zone")))
    .withColumn("delay_hr", F.col("duration_hr") - F.col("miles") / F.col("ff_eff")))

rq1 = (trip_delay
    .groupBy("pu", "hour", "daytype")          
    .agg(F.sum("revenue").alias("rev"),
         F.sum("duration_hr").alias("hours"),
         F.sum("miles").alias("miles"),
         F.sum("delay_hr").alias("delay_hr"),
         F.count(F.lit(1)).alias("trips"))
    .withColumn("yield_hr", F.col("rev") / F.col("hours"))
    .withColumn("speed", F.col("miles") / F.col("hours"))
    .join(F.broadcast(ff_zone.select("pu", "ff_yield_zone")), ["pu"])
    .withColumn("yield_gap", F.col("ff_yield_zone") - F.col("yield_hr"))
    .withColumn("forgone", F.col("yield_gap") * F.col("hours"))
    .cache())

top = (rq1.filter("daytype = 'weekday'")
          .join(F.broadcast(zones.withColumnRenamed("loc", "pu")), "pu")
          .orderBy(F.desc("forgone")))
top.select("zone", "borough", "hour", "trips", "speed",
           "yield_hr", "ff_yield_zone", "delay_hr", "forgone").show(25, truncate=False)

weekday_forgone = rq1.filter("daytype = 'weekday'").agg(F.sum("forgone")).first()[0]
print(f"total forgone, weekdays 2022-2025: ${weekday_forgone:,.0f}")

In [ ]:
zones_pu = F.broadcast(zones.withColumnRenamed("loc", "pu"))
wd = rq1.filter("daytype = 'weekday'")

print("=== all zones (airport-dominated) ===")
wd.join(zones_pu, "pu").orderBy(F.desc("forgone")).select(
    "zone", "borough", "hour", "trips", "speed", "yield_hr", "ff_yield_zone", "forgone"
).show(10, truncate=False)

print("=== excluding JFK and LaGuardia ===")
wd.filter("pu not in (132, 138)").join(zones_pu, "pu").orderBy(F.desc("forgone")).select(
    "zone", "borough", "hour", "trips", "speed", "yield_hr", "ff_yield_zone", "forgone"
).show(20, truncate=False)

f_all = wd.agg(F.sum("forgone")).first()[0]
f_noair = wd.filter("pu not in (132, 138)").agg(F.sum("forgone")).first()[0]
print(f"\nforgone, all zones: ${f_all:,.0f}")
print(f"forgone, no airports: ${f_noair:,.0f}  ({f_noair / f_all:.0%} of total)")

## 04: RDD Twin

In [ ]:
def timed(label, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    print(f"{label:34s} {dt:8.1f}s   rows={len(out):,}")
    return out, dt

def dataframe_version():
    return (filtered.groupBy("pu", "hour", "daytype")
                   .agg(F.sum("revenue").alias("rev"),
                        F.sum("duration_hr").alias("hours"),
                        F.count(F.lit(1)).alias("trips"))
                   .collect())

def rdd_version():
    rows  = filtered.select("pu", "hour", "daytype", "revenue", "duration_hr").rdd
    pairs = rows.map(lambda r: ((r["pu"], r["hour"], r["daytype"]),
                                (r["revenue"], r["duration_hr"], 1)))              # MAP
    sums  = pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2]))  # SHUFFLE + REDUCE
    return sums.map(lambda kv: kv[0] + kv[1]).collect()

df_out,  t_df  = timed("DataFrame  groupBy/agg", dataframe_version)
rdd_out, t_rdd = timed("RDD        map/reduceByKey", rdd_version)
print(f"\nRDD is {t_rdd / t_df:.1f}x the DataFrame wall-clock")

a = {(r["pu"], r["hour"], r["daytype"]): round(r["rev"], 2) for r in df_out}
b = {(t[0], t[1], t[2]): round(t[3], 2) for t in rdd_out} # (pu, hour, daytype, rev, hours, trips)
print("same keys:", set(a) == set(b), "| same sums:", a == b)

In [ ]:
one = spark.read.parquet(str(LOCAL_CUR / "year=2024" / "month=6"))
pairs_one = (one.select("pu", "hour", "daytype", "revenue", "duration_hr").rdd
                .map(lambda r: ((r["pu"], r["hour"], r["daytype"]),
                                (r["revenue"], r["duration_hr"], 1))))

def with_reduce():
    return pairs_one.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2])).collect()

def with_group():
    return (pairs_one.groupByKey()
                     .mapValues(lambda vs: tuple(map(sum, zip(*vs))))
                     .collect())

_, t_reduce = timed("reduceByKey (map-side combine)", with_reduce)
_, t_group  = timed("groupByKey (no combine)", with_group)

## 05: Execution Evidence

In [ ]:
def capture_plan(df, name):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain("formatted")
    text = buf.getvalue()
    (LOCAL_OUT / f"plan_{name}.txt").write_text(text)
    print(text[:1800])
    return text

_ = capture_plan(rq1, "rq1_zone_hour")
_ = capture_plan(rq2, "rq2_route_window")

In [ ]:
def _event_files():
    entries = sorted(EVENTS.iterdir(), key=lambda p: p.stat().st_mtime)
    assert entries, f"no event logs in {EVENTS}"
    latest = entries[-1]
    if latest.is_dir():
        return sorted((f for f in latest.iterdir() if f.name.startswith("events_")),
                      key=lambda p: p.stat().st_mtime)
    return [latest]

def _lines(path):
    if ".zstd" in path.name:
        import zstandard as zstd
        with open(path, "rb") as fh, zstd.ZstdDecompressor().stream_reader(fh) as r:
            yield from io.TextIOWrapper(r, encoding="utf-8")
    else:
        with open(path, encoding="utf-8") as fh:
            yield from fh

def stage_metrics():
    rows = []
    for path in _event_files():
        for line in _lines(path):
            if not line.strip():
                continue
            e = json.loads(line)
            if e.get("Event") != "SparkListenerTaskEnd":
                continue
            m  = e.get("Task Metrics") or {}
            sr = m.get("Shuffle Read Metrics") or {}
            rows.append({
                "stage": e["Stage ID"],
                "task_ms": m.get("Executor Run Time", 0),
                "in_mb": (m.get("Input Metrics") or {}).get("Bytes Read", 0) / 2**20,
                "sh_w_mb": (m.get("Shuffle Write Metrics") or {}).get("Shuffle Bytes Written", 0) / 2**20,
                "sh_r_mb": (sr.get("Remote Bytes Read", 0) + sr.get("Local Bytes Read", 0)) / 2**20,
            })
    t = pd.DataFrame(rows)

    out = (t.groupby("stage")
             .agg(tasks=("task_ms", "size"),
                  median_ms=("task_ms", "median"),
                  max_ms=("task_ms", "max"),
                  input_mb=("in_mb", "sum"),
                  shuffle_write_mb=("sh_w_mb", "sum"),
                  shuffle_read_mb=("sh_r_mb", "sum"))
             .reset_index())
    den = out.median_ms.astype("float64")  
    out["skew_ratio"] = (out.max_ms / den.where(den > 0)).round(1)
    return out

metrics = stage_metrics()
metrics.to_csv(LOCAL_OUT / "stage_metrics.csv", index=False)
print(metrics.to_string(index=False))

## 06: Robustness

In [ ]:
AIRPORTS = [132, 138] # JFK, LaGuardia

variants = {
    "headline": filtered,
    "no_airports": filtered.filter("pu not in (132, 138) AND do not in (132, 138)"),
    "pre_2025_only": filtered.filter("year < 2025"),
    "2025_only": filtered.filter("year = 2025"),
}

summary = []
for name, frame in variants.items():
    agg = (frame.groupBy("hour")
                .agg((F.sum("miles") / F.sum("duration_hr")).alias("speed"),
                     (F.sum("revenue") / F.sum("duration_hr")).alias("yield_hr"),
                     F.count(F.lit(1)).alias("trips"))
                .orderBy("hour").toPandas())
    summary.append({
        "variant": name,
        "trips": int(agg.trips.sum()),
        "slowest_hour": int(agg.loc[agg.speed.idxmin(), "hour"]),
        "min_speed": round(agg.speed.min(), 2),
        "peak_yield": round(agg.yield_hr.max(), 2),
        "trough_yield": round(agg.yield_hr.min(), 2),
    })

sens = pd.DataFrame(summary)
sens.to_csv(LOCAL_OUT / "sensitivity.csv", index=False)
print(sens.to_string(index=False))

## 07: Handover

In [ ]:
checks, failures = [], []

def check(name, condition, detail=""):
    checks.append((name, bool(condition), detail))
    if not condition:
        failures.append(name)

# 00 — does the filtered data match what Part 1 established?
prof = (filtered.groupBy("hour")
        .agg(F.count(F.lit(1)).alias("trips"),
             (F.sum("miles") / F.sum("duration_hr")).alias("speed"))
        .orderBy("hour").toPandas())
peak_hour    = int(prof.loc[prof.trips.idxmax(), "hour"])
slowest_hour = int(prof.loc[prof.speed.idxmin(), "hour"])
retention    = filtered.count() / audit["rows_read"]

check("J0 peak hour is 18:00 (timezone sane)", peak_hour == 18, f"got {peak_hour}")
check("J0 slowest hour is in the afternoon peak", 15 <= slowest_hour <= 18, f"got {slowest_hour}")
check("J0 retention 93-98%",       0.93 <= retention <= 0.98, f"got {retention:.2%}")

# 01 — is the baseline plausible and well supported?
ff_stats = ff_route.agg(F.min("ff_speed"), F.max("ff_speed"), F.count(F.lit(1))).first()
check("J1 baseline has 1k+ routes", ff_stats[2] >= 1000, f"got {ff_stats[2]:,}")
check("J1 night speeds are 5-60 mph", ff_stats[0] > 5 and ff_stats[1] < 60,
      f"{ff_stats[0]:.1f} to {ff_stats[1]:.1f} mph")

# 02 — the two rankings must disagree, or the support filter is wrong
pm_pd  = pm.select("pu", "do", "tti", "delay_hr_total").toPandas()
by_tti = set(map(tuple, pm_pd.nlargest(20, "tti")[["pu", "do"]].values))
by_hrs = set(map(tuple, pm_pd.nlargest(20, "delay_hr_total")[["pu", "do"]].values))
check("J2 severity and burden rankings differ", by_tti != by_hrs,
      f"{len(by_tti & by_hrs)} of 20 shared")

# 03 — shape and sign
n_cells = rq1.count()
check("J3 produces <= 12,720 cells", n_cells <= 12720, f"got {n_cells:,}")
check("J3 has negative gaps (honest baseline)",
      rq1.filter("yield_gap < 0").count() > 0)

# 04 — the twin agrees
check("J4 RDD and DataFrame agree", set(a) == set(b) and a == b)

# 05 — evidence exists
check("J5 stage metrics captured", len(metrics) > 0, f"{len(metrics)} stages")
check("J5 a shuffle actually happened", metrics.shuffle_write_mb.sum() > 0,
      f"{metrics.shuffle_write_mb.sum():.1f} MB written")

for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"[{detail}]" if detail else ""))
print("\n", "ALL PASS" if not failures else f"{len(failures)} FAILED: {failures}")

In [ ]:
named_pd = named.toPandas()
rq1_pd   = rq1.join(zones_pu, "pu").toPandas()
named_pd.to_csv(DRIVE_OUT / "rq2_route_window.csv", index=False)
rq1_pd.to_csv(DRIVE_OUT / "rq1_zone_hour.csv", index=False)
print(f"rq2: {len(named_pd):,} rows | rq1: {len(rq1_pd):,} rows")

big = metrics[(metrics.tasks > 1) &
              ((metrics.shuffle_write_mb > 1) | (metrics.input_mb > 1) | (metrics.median_ms > 500))]
big.to_csv(DRIVE_OUT / "stage_metrics_significant.csv", index=False)

c = metrics[(metrics.tasks == 8) & (metrics.shuffle_write_mb > 0)].shuffle_write_mb.sort_values().tolist()
assert len(c) == 2, f"expected 2 combiner stages, got {c}"
reduce_mb, group_mb = c[0], c[-1]

n_filtered = filtered.count()
part2 = {
    "months": len(staged),
    "working_gib": round(nbytes / 2**30, 2),
    "filtered_gib": round(sum(f.stat().st_size for f in LOCAL_CUR.rglob("*.parquet")) / 2**30, 2),
    "rows_read": int(audit["rows_read"]),
    "rows_filtered": int(n_filtered),
    "retention_pct": round(100 * n_filtered / audit["rows_read"], 2),
    "baseline_routes": int(ff_route.count()),
    "route_window_cells": int(rq2.count()),
    "forgone_all": round(float(f_all)),
    "forgone_no_airports": round(float(f_noair)),
    "slowest_hour": int(slowest_hour),
    "t_dataframe_s": round(t_df, 1),
    "t_rdd_s": round(t_rdd, 1),
    "rdd_penalty_x": round(t_rdd / t_df, 1),
    "reduce_shuffle_mb": round(reduce_mb, 3),
    "group_shuffle_mb": round(group_mb, 3),
    "max_skew_ratio": float(metrics[metrics.median_ms >= 500].skew_ratio.max()),
}
(DRIVE_OUT / "part2_summary.json").write_text(json.dumps(part2, indent=2))

report = [
    ("Months",  f"{part2['months']}"),
    ("Raw catalogue",   "~31 GB"),
    ("Subset on disk",  f"{part2['working_gib']:.2f} GiB"),
    ("Filtered (processed)",    f"{part2['filtered_gib']:.2f} GiB"),
    ("Rows read",   f"{part2['rows_read']:,}"),
    ("Rows retained",   f"{part2['rows_filtered']:,}"),
    ("Retention",   f"{part2['retention_pct']:.2f}\\%"),
    ("Baseline routes", f"{part2['baseline_routes']:,}"),
    ("Route-window cells",  f"{part2['route_window_cells']:,}"),
    ("Forgone, all zones",  f"\\${part2['forgone_all']/1e6:,.1f}M"),
    ("Forgone, excl. airports", f"\\${part2['forgone_no_airports']/1e6:,.1f}M"),
    ("Slowest hour",    f"{part2['slowest_hour']}:00"),
    ("DataFrame runtime",   f"{part2['t_dataframe_s']:.1f} s"),
    ("RDD runtime", f"{part2['t_rdd_s']:.1f} s"),
    ("RDD penalty", f"{part2['rdd_penalty_x']:.0f}$\\times$"),
    ("reduceByKey shuffle write",   f"{part2['reduce_shuffle_mb']:.3f} MB"),
    ("groupByKey shuffle write",    f"{part2['group_shuffle_mb']:.3f} MB"),
    ("Combiner reduction",  f"{group_mb/reduce_mb:.0f}$\\times$"),
    ("Max skew ratio",  f"{part2['max_skew_ratio']:.1f}$\\times$"),
]
pd.DataFrame(report, columns=["metric", "value"]).to_csv(DRIVE_OUT / "part2_summary.csv", index=False)
for k, v in report:
    print(f"{k:28s} {v}")

In [ ]:
spark.stop()
print("event logs in", EVENTS, "-", len(list(EVENTS.iterdir())), "run(s) recorded")